# Exploitation Zone: GPU-First Image Vectorization

This notebook is the first-version source code for the image vectorization flow. The Airflow DAG should be implemented from this notebook logic later, not imported by this notebook.

Spark stays responsible for Trusted Zone Delta catalogue IO, filtering, id generation, and status catalogue writes. CLIP inference and Milvus upsert run in the notebook Python process so CUDA can be used when available, with CPU fallback when it is not.

Pipeline:

```text
trusted-zone/file_catalog
        -> Spark Delta catalogue read/filter
        -> Python batch loader
        -> MinIO image read
        -> CLIP image embeddings on cuda or cpu
        -> pymilvus upsert
        -> Spark Delta status catalogue write
```

In [1]:
import io
import os
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3
import pandas as pd
import torch
from PIL import Image
from pymilvus import DataType, MilvusClient
from pyspark.sql import SparkSession, functions as F
from transformers import CLIPModel, CLIPProcessor

os.environ.setdefault('MINIO_ENDPOINT', 'http://minio:9000')
MINIO_ROLE = 'writer'
os.environ.setdefault('MINIO_WRITER_ACCESS_KEY', 'bdm_writer')
os.environ.setdefault('MINIO_WRITER_SECRET_KEY', 'bdm_writer_password')
os.environ.setdefault('SPARK_MASTER_URL', 'spark://spark-master:7077')
os.environ.setdefault('TRUSTED_IMAGE_CATALOGUE_PATH', 's3a://trusted-zone/file_catalog/')
os.environ.setdefault('IMAGE_EMBEDDING_STATUS_PATH', 's3a://trusted-zone/embedding_status/image_clip/')
os.environ.setdefault('MILVUS_URI', 'http://milvus:19530')
os.environ.setdefault('MILVUS_IMAGE_COLLECTION', 'image_vector_catalog')
os.environ.setdefault('CLIP_MODEL_PATH', 'openai/clip-vit-base-patch32')
os.environ.setdefault('HF_HOME', '/opt/spark/hf_cache')
os.environ.setdefault('IMAGE_EMBEDDING_DEVICE', 'auto')
os.environ.setdefault('IMAGE_EMBEDDING_BATCH_SIZE', '32')
os.environ.setdefault('IMAGE_CATALOGUE_SPARK_PARTITIONS', '8')

'8'

## 1. Runtime Check

A CUDA PyTorch build is installed in the notebook image. The pipeline still checks CUDA availability at runtime before moving CLIP to the GPU.

In [2]:
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda runtime:', torch.version.cuda)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch: 2.7.0+cu128
cuda available: True
cuda runtime: 12.8
gpu: NVIDIA GeForce RTX 5070


## 2. Spark Catalogue Helpers

These helpers create the Spark session, read the Trusted Delta image catalogue, generate a stable global `id`, skip already-successful embeddings, and produce the pending image list.

In [3]:
@dataclass
class ImageVectorizationConfig:
    catalogue_path: str = os.getenv('TRUSTED_IMAGE_CATALOGUE_PATH')
    status_catalogue_path: str = os.getenv('IMAGE_EMBEDDING_STATUS_PATH')
    collection_name: str = os.getenv('MILVUS_IMAGE_COLLECTION')
    milvus_uri: str = os.getenv('MILVUS_URI')
    model_name_or_path: str = os.getenv('CLIP_MODEL_PATH')
    embedding_dim: int = 512
    batch_size: int = int(os.getenv('IMAGE_EMBEDDING_BATCH_SIZE', '32'))
    spark_partitions: int = int(os.getenv('IMAGE_CATALOGUE_SPARK_PARTITIONS', '8'))
    max_images: int | None = None
    skip_existing: bool = True
    device: str = os.getenv('IMAGE_EMBEDDING_DEVICE', 'auto')


def storage_options():
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f'MINIO_{role_prefix}_ACCESS_KEY')
    secret_key = os.getenv(f'MINIO_{role_prefix}_SECRET_KEY')
    if not access_key or not secret_key:
        raise RuntimeError(f'Missing MinIO {MINIO_ROLE} credentials in environment')
    return {
        'AWS_ACCESS_KEY_ID': access_key,
        'AWS_SECRET_ACCESS_KEY': secret_key,
        'AWS_ENDPOINT_URL': os.getenv('MINIO_ENDPOINT'),
        'AWS_S3_ALLOW_UNSAFE_RENAME': 'true',
        'AWS_S3_ADDRESSING_STYLE': 'path',
        'AWS_ALLOW_HTTP': 'true',
        'region': 'us-east-1',
    }


def ensure_java_home():
    if os.getenv('JAVA_HOME'):
        return
    for candidate in ('/usr/lib/jvm/java-17-openjdk-amd64', '/usr/lib/jvm/java-11-openjdk-amd64', '/usr/lib/jvm/default-java'):
        if os.path.exists(os.path.join(candidate, 'bin', 'java')):
            os.environ['JAVA_HOME'] = candidate
            os.environ['PATH'] = f"{candidate}/bin:{os.environ.get('PATH', '')}"
            return
    java_path = shutil.which('java')
    if java_path:
        os.environ['JAVA_HOME'] = os.path.dirname(os.path.dirname(os.path.realpath(java_path)))


def create_spark_session(app_name: str, config: ImageVectorizationConfig):
    ensure_java_home()
    opts = storage_options()
    packages = ','.join([
        'org.apache.hadoop:hadoop-aws:3.3.4',
        'com.amazonaws:aws-java-sdk-bundle:1.12.262',
        f"io.delta:delta-spark_2.13:{os.getenv('DELTA_SPARK_VERSION', '4.1.0')}",
    ])
    spark = (
        SparkSession.builder.appName(app_name)
        .master(os.getenv('SPARK_MASTER_URL', 'spark://spark-master:7077'))
        .config('spark.jars.packages', packages)
        .config('spark.sql.shuffle.partitions', str(config.spark_partitions))
        .config('spark.ui.showConsoleProgress', 'false')
        .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.hadoop.fs.s3a.endpoint', opts['AWS_ENDPOINT_URL'])
        .config('spark.hadoop.fs.s3a.path.style.access', 'true')
        .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
        .config('spark.hadoop.fs.s3a.access.key', opts['AWS_ACCESS_KEY_ID'])
        .config('spark.hadoop.fs.s3a.secret.key', opts['AWS_SECRET_ACCESS_KEY'])
        .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel(os.getenv('SPARK_LOG_LEVEL', 'ERROR'))
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    for item in hadoop_conf.iterator():
        key, value = item.getKey(), item.getValue()
        if isinstance(value, str) and (value.endswith('s') or value.endswith('h')):
            digits = ''.join(char for char in value if char.isdigit())
            if digits:
                hadoop_conf.set(key, digits)
    return spark


def resolve_column(columns, candidates, required=True):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    if required:
        raise ValueError(f'None of the expected columns exist: {candidates}. Available columns: {sorted(columns)}')
    return None


def build_pending_catalogue_dataframe(spark, config: ImageVectorizationConfig):
    df = spark.read.format('delta').load(config.catalogue_path)
    columns = set(df.columns)
    path_column = resolve_column(columns, ['trusted_path', 'image_path', 'file_path', 'path'])
    label_column = resolve_column(columns, ['label', 'source_type'], required=False)

    if 'file_type' in columns:
        df = df.filter(F.lower(F.col('file_type')) == F.lit('image'))
    if 'is_corrupted' in columns:
        df = df.filter(F.col('is_corrupted') == F.lit(False))

    id_parts = [name for name in ('id', 'file_id', 'landing_file_id', path_column) if name in columns]
    stable_key = F.concat_ws('||', *[F.coalesce(F.col(name).cast('string'), F.lit('')) for name in id_parts])
    pending = (
        df.filter(F.col(path_column).isNotNull())
        .filter(F.length(F.trim(F.col(path_column).cast('string'))) > 0)
        .withColumn('id', F.sha2(stable_key, 256))
        .withColumn('image_path', F.col(path_column).cast('string'))
        .withColumn('label', F.col(label_column).cast('string') if label_column else F.lit(''))
        .select('id', 'image_path', 'label')
        .dropDuplicates(['id'])
    )

    if config.skip_existing:
        try:
            existing = (
                spark.read.format('delta')
                .load(config.status_catalogue_path)
                .filter(F.col('status') == F.lit('SUCCESS'))
                .select('id')
                .distinct()
            )
            pending = pending.join(existing, on='id', how='left_anti')
        except Exception as exc:
            print(f'No readable embedding status catalogue yet: {exc}')

    if config.max_images:
        pending = pending.limit(config.max_images)
    return pending.repartition(config.spark_partitions)


def iter_catalogue_batches(dataframe, batch_size):
    batch = []
    for row in dataframe.toLocalIterator():
        batch.append({'id': row['id'], 'image_path': row['image_path'], 'label': row['label'] or ''})
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

## 3. Python Batch Embedding Helpers

These helpers run outside Spark executors: they read images from MinIO, load CLIP on GPU or CPU, upsert vectors to Milvus, and build per-image status records.

In [4]:
def parse_s3_path(path: str):
    parsed = urlparse(path)
    if parsed.scheme in {'s3', 's3a'}:
        return parsed.netloc, parsed.path.lstrip('/')
    raise ValueError(f'Expected s3/s3a image path, got: {path}')


def s3_client():
    opts = storage_options()
    return boto3.client(
        's3',
        endpoint_url=opts['AWS_ENDPOINT_URL'],
        aws_access_key_id=opts['AWS_ACCESS_KEY_ID'],
        aws_secret_access_key=opts['AWS_SECRET_ACCESS_KEY'],
    )


def read_image(client, image_path: str):
    bucket, key = parse_s3_path(image_path)
    response = client.get_object(Bucket=bucket, Key=key)
    try:
        content = response['Body'].read()
    finally:
        response['Body'].close()
    return Image.open(io.BytesIO(content)).convert('RGB')


def load_clip_model(config: ImageVectorizationConfig):
    requested = config.device.lower()
    if requested == 'auto':
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    else:
        device = requested
    if device == 'cuda' and not torch.cuda.is_available():
        print('CUDA requested but unavailable; using CPU')
        device = 'cpu'
    if device not in {'cuda', 'cpu'}:
        raise ValueError(f'Unsupported device: {config.device}')

    processor = CLIPProcessor.from_pretrained(config.model_name_or_path)
    model = CLIPModel.from_pretrained(config.model_name_or_path).to(device)
    model.eval()
    print(f'Loaded CLIP from {config.model_name_or_path} on {device}')
    return processor, model, device


def encode_images(images, processor, model, device):
    inputs = processor(images=images, return_tensors='pt', padding=True)
    inputs = {name: value.to(device) for name, value in inputs.items()}
    with torch.inference_mode():
        features = model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return features.detach().cpu().float().numpy().tolist()


def is_cuda_failure(exc):
    message = str(exc).lower()
    return 'cuda' in message or 'cudnn' in message or 'out of memory' in message


def ensure_milvus_collection(config: ImageVectorizationConfig):
    client = MilvusClient(uri=config.milvus_uri)
    if client.has_collection(config.collection_name):
        client.load_collection(config.collection_name)
        return

    schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
    schema.add_field(field_name='id', datatype=DataType.VARCHAR, is_primary=True, max_length=128)
    schema.add_field(field_name='image_path', datatype=DataType.VARCHAR, max_length=1024)
    schema.add_field(field_name='label', datatype=DataType.VARCHAR, max_length=256)
    schema.add_field(field_name='embeddings', datatype=DataType.FLOAT_VECTOR, dim=config.embedding_dim)

    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name='embeddings',
        index_type=os.getenv('MILVUS_IMAGE_INDEX_TYPE', 'IVF_FLAT'),
        metric_type=os.getenv('MILVUS_IMAGE_METRIC_TYPE', 'COSINE'),
        params={'nlist': int(os.getenv('MILVUS_IMAGE_INDEX_NLIST', '128'))},
    )
    client.create_collection(collection_name=config.collection_name, schema=schema, index_params=index_params)
    client.load_collection(config.collection_name)


def write_milvus(config: ImageVectorizationConfig, rows):
    if not rows:
        return
    client = MilvusClient(uri=config.milvus_uri)
    client.upsert(collection_name=config.collection_name, data=rows)
    client.flush(collection_name=config.collection_name)
    client.load_collection(collection_name=config.collection_name)


def status_record(row, status, error_msg, vector_dim, device, config: ImageVectorizationConfig):
    return {
        'id': row['id'],
        'image_path': row['image_path'],
        'label': row.get('label', ''),
        'status': status,
        'error_msg': str(error_msg)[:2048],
        'vector_dim': int(vector_dim),
        'device': device,
        'model_name': config.model_name_or_path,
        'collection_name': config.collection_name,
        'processed_at': datetime.now(timezone.utc).isoformat(),
    }


def process_image_batch(rows, processor, model, device, config: ImageVectorizationConfig):
    client = s3_client()
    images = []
    valid_rows = []
    statuses = []
    for row in rows:
        try:
            images.append(read_image(client, row['image_path']))
            valid_rows.append(row)
        except Exception as exc:
            statuses.append(status_record(row, 'FAILED_IMAGE_READ', exc, 0, device, config))

    if not valid_rows:
        return statuses, device

    try:
        embeddings = encode_images(images, processor, model, device)
    except Exception as exc:
        if device == 'cuda' and is_cuda_failure(exc):
            print(f'CUDA batch failed, retrying on CPU: {exc}')
            torch.cuda.empty_cache()
            model.to('cpu')
            device = 'cpu'
            embeddings = encode_images(images, processor, model, device)
        else:
            return [status_record(row, 'FAILED_EMBEDDING', exc, 0, device, config) for row in valid_rows], device

    payload = [
        {
            'id': row['id'],
            'image_path': row['image_path'],
            'label': row['label'] or '',
            'embeddings': embedding,
        }
        for row, embedding in zip(valid_rows, embeddings)
    ]
    try:
        write_milvus(config, payload)
        statuses.extend(status_record(row, 'SUCCESS', '', config.embedding_dim, device, config) for row in valid_rows)
    except Exception as exc:
        statuses.extend(status_record(row, 'FAILED_MILVUS_WRITE', exc, 0, device, config) for row in valid_rows)
    return statuses, device


def write_status_catalogue(spark, statuses, config: ImageVectorizationConfig):
    if not statuses:
        print('No embedding statuses to write')
        return
    status_df = spark.createDataFrame(pd.DataFrame(statuses))
    status_df.write.format('delta').mode('append').option('mergeSchema', 'true').save(config.status_catalogue_path)
    print(f'Wrote {len(statuses)} status records to {config.status_catalogue_path}')

## 4. Configure The Run

Set `max_images` to a small number for a smoke test, or leave it as `None` for the full trusted catalogue.

In [5]:
config = ImageVectorizationConfig(
    max_images=None,
    skip_existing=True,
    device=os.getenv('IMAGE_EMBEDDING_DEVICE', 'auto'),
)

config

ImageVectorizationConfig(catalogue_path='s3a://trusted-zone/file_catalog/', status_catalogue_path='s3a://trusted-zone/embedding_status/image_clip/', collection_name='image_vector_catalog', milvus_uri='http://milvus:19530', model_name_or_path='openai/clip-vit-base-patch32', embedding_dim=512, batch_size=32, spark_partitions=8, max_images=None, skip_existing=True, device='auto')

## 5. Run Vectorization

Spark is active only for catalogue IO. Model inference and Milvus writes happen in this notebook kernel.

In [6]:
spark = create_spark_session('exploitation_image_vectorization', config)
try:
    pending_df = build_pending_catalogue_dataframe(spark, config)
    ensure_milvus_collection(config)
    processor, model, device = load_clip_model(config)

    status_buffer = []
    summary = {'success': 0, 'failed': 0, 'total': 0}
    for batch in iter_catalogue_batches(pending_df, config.batch_size):
        batch_statuses, device = process_image_batch(batch, processor, model, device, config)
        status_buffer.extend(batch_statuses)
        for item in batch_statuses:
            summary['total'] += 1
            if item['status'] == 'SUCCESS':
                summary['success'] += 1
            else:
                summary['failed'] += 1

    write_status_catalogue(spark, status_buffer, config)
finally:
    spark.stop()

summary

Loaded CLIP from openai/clip-vit-base-patch32 on cuda
No embedding statuses to write


{'success': 0, 'failed': 0, 'total': 0}

## 6. Inspect The Status Catalogue

The status catalogue stays Delta-native so the downstream Airflow version can keep the same catalogue contract.

In [7]:
spark = create_spark_session('exploitation_image_status_check', config)
try:
    status_df = spark.read.format('delta').load(config.status_catalogue_path)
    status_df.groupBy('status', 'device').count().orderBy('status', 'device').show(truncate=False)
    status_df.orderBy('processed_at', ascending=False).select(
        'id', 'image_path', 'status', 'device', 'vector_dim', 'error_msg'
    ).show(20, truncate=80)
finally:
    spark.stop()

+-------+------+-----+
|status |device|count|
+-------+------+-----+
|SUCCESS|cuda  |6674 |
+-------+------+-----+

+----------------------------------------------------------------+----------------------------------------------------------------------------+-------+------+----------+---------+
|                                                              id|                                                                  image_path| status|device|vector_dim|error_msg|
+----------------------------------------------------------------+----------------------------------------------------------------------------+-------+------+----------+---------+
|601c58f70c474162882e14e4a549adc9aa596473532579bd5a877fa0756e987f|s3a://trusted-zone/unstructured/image/image_weather_10_463_1779642580000.png|SUCCESS|  cuda|       512|         |
|b09b2629556cb59335b5eb8829345b95e2fd6ad2c64b7451be137bc5f3068256| s3a://trusted-zone/unstructured/image/image_weather_9_684_1779642580000.png|SUCCESS|  cuda|      

## 7. Milvus Smoke Test

Verify that the collection is reachable and contains inserted vectors.

In [8]:
client = MilvusClient(uri=config.milvus_uri)
client.flush(collection_name=config.collection_name)
client.load_collection(collection_name=config.collection_name)
print('collection:', config.collection_name)
print('rows:', client.get_collection_stats(config.collection_name).get('row_count'))

collection: image_vector_catalog
rows: 6674
